In [2]:
import numpy as np
import pandas as pd
from numpy.linalg import norm

## Information retrieval: Option 1 (TF-IDF) and Option 3 (cosine ranking)

We treat each **document** as a row of numbers: how often each **term** from a fixed **vocabulary** appears (a *bag of words*). A **query** lists a few terms we want to find.

This notebook does two related things:

1. **Option 1** — Build **TF-IDF** (with **log** term frequency), then score each document with a **plain dot product** against the query (sum of TF-IDF on query terms). The output table has that score in the first column and **raw TF-IDF** in the term columns.

2. **Option 3** — **L2-normalize** document and query vectors, then rank by **cosine similarity** (dot product of unit vectors). The ranked table and the wide table use **normalized** TF-IDF in the term columns.

In [3]:
np.random.seed(42)

terms = [f"t{i}" for i in range(10)]
docs = [f"D{i}" for i in range(20)]

# Create term frequencies (0–10 occurrences)
data = np.random.randint(0, 11, size=(20, 10))

# Make some terms appear in ALL documents (simulate common words)
data[:, 0] += 5  # t0 very common
data[:, 1] += 3  # t1 common

df = pd.DataFrame(data, index=docs, columns=terms)

print("Input Term Frequencies:")
df

Input Term Frequencies:


,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
D0,11,6,10,7,4,6,9,2,6,10
D1,15,10,4,3,7,7,2,5,4,1
D2,12,8,1,4,0,9,5,8,0,10
D3,15,12,2,6,3,8,2,4,2,6
D4,9,11,6,1,3,8,1,9,8,9
D5,9,4,3,6,7,2,0,3,1,7
D6,8,4,5,5,9,3,5,1,9,1
D7,14,6,7,6,8,7,4,1,4,7
D8,14,11,8,0,8,6,8,7,0,7
D9,12,13,2,0,7,2,2,0,10,4


In [4]:
query_terms = ["t1", "t3", "t7"]
query_vector = np.array([1 if term in query_terms else 0 for term in terms])

print("Query Vector:")
print(query_vector)

Query Vector:
[0 1 0 1 0 0 0 1 0 0]


In [5]:
tf = np.log1p(df)
tf_df = pd.DataFrame(tf, index=docs, columns=terms)

print("TF (log-scaled):")
tf_df

TF (log-scaled):


,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
D0,2.484907,1.945910,2.397895,2.079442,1.609438,1.945910,2.302585,1.098612,1.945910,2.397895
D1,2.772589,2.397895,1.609438,1.386294,2.079442,2.079442,1.098612,1.791759,1.609438,0.693147
D2,2.564949,2.197225,0.693147,1.609438,0.000000,2.302585,1.791759,2.197225,0.000000,2.397895
D3,2.772589,2.564949,1.098612,1.945910,1.386294,2.197225,1.098612,1.609438,1.098612,1.945910
D4,2.302585,2.484907,1.945910,0.693147,1.386294,2.197225,0.693147,2.302585,2.197225,2.302585
D5,2.302585,1.609438,1.386294,1.945910,2.079442,1.098612,0.000000,1.386294,0.693147,2.079442
D6,2.197225,1.609438,1.791759,1.791759,2.302585,1.386294,1.791759,0.693147,2.302585,0.693147
D7,2.708050,1.945910,2.079442,1.945910,2.197225,2.079442,1.609438,0.693147,1.609438,2.079442
D8,2.708050,2.484907,2.197225,0.000000,2.197225,1.945910,2.197225,2.079442,0.000000,2.079442
D9,2.564949,2.639057,1.098612,0.000000,2.079442,1.098612,1.098612,0.000000,2.397895,1.609438


In [6]:
N = len(df)

df_count = (df > 0).sum(axis=0)
idf = np.log(N / df_count)

idf_df = pd.DataFrame(idf).T
idf_df.columns = terms

print("IDF:")
idf_df

IDF:


,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
0,0.0,0.0,0.105361,0.162519,0.051293,0.0,0.162519,0.105361,0.356675,0.105361


In [7]:
tfidf = tf * idf
tfidf_df = pd.DataFrame(tfidf, index=docs, columns=terms)

print("TF-IDF Matrix:")
tfidf_df

TF-IDF Matrix:


,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
D0,0.0,0.0,0.252643,0.337949,0.082553,0.0,0.374214,0.115750,0.694057,0.252643
D1,0.0,0.0,0.169571,0.225299,0.106661,0.0,0.178545,0.188781,0.574046,0.073030
D2,0.0,0.0,0.073030,0.261564,0.000000,0.0,0.291195,0.231501,0.000000,0.252643
D3,0.0,0.0,0.115750,0.316247,0.071108,0.0,0.178545,0.169571,0.391847,0.205022
D4,0.0,0.0,0.205022,0.112650,0.071108,0.0,0.112650,0.242602,0.783695,0.242602
D5,0.0,0.0,0.146061,0.316247,0.106661,0.0,0.000000,0.146061,0.247228,0.219091
D6,0.0,0.0,0.188781,0.291195,0.118107,0.0,0.291195,0.073030,0.821274,0.073030
D7,0.0,0.0,0.219091,0.316247,0.112703,0.0,0.261564,0.073030,0.574046,0.219091
D8,0.0,0.0,0.231501,0.000000,0.112703,0.0,0.357091,0.219091,0.000000,0.219091
D9,0.0,0.0,0.115750,0.000000,0.106661,0.0,0.178545,0.000000,0.855269,0.169571


In [ ]:
# Option 1: query relevance without cosine - dot product of TF-IDF row and binary query.
# For each document d: score(d) = sum of TF-IDF(d,t) over query terms t.
tfidf_query_scores = tfidf @ query_vector

option1_output = tfidf_df.copy()
option1_output.insert(0, "Query_TF-IDF_score", tfidf_query_scores)

print("Option 1 output (spec): first column = query relevance; other columns = TF-IDF per term")
option1_output

In [8]:
# Option 3: scale each document vector to unit length (L2 norm = 1) for cosine similarity.
tfidf_norm = tfidf / norm(tfidf, axis=1, keepdims=True)
tfidf_norm = np.nan_to_num(tfidf_norm)

tfidf_norm_df = pd.DataFrame(tfidf_norm, index=docs, columns=terms)

print("L2-normalized TF-IDF (unit vectors; used for cosine with the query)")
tfidf_norm_df

Normalized TF-IDF:


,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
D0,0.0,0.0,0.268734,0.359473,0.087811,0.0,0.398048,0.123123,0.738262,0.268734
D1,0.0,0.0,0.241440,0.320787,0.151868,0.0,0.254218,0.268791,0.817343,0.103983
D2,0.0,0.0,0.139020,0.497910,0.000000,0.0,0.554315,0.440682,0.000000,0.480929
D3,0.0,0.0,0.189101,0.516651,0.116168,0.0,0.291688,0.277028,0.640159,0.334943
D4,0.0,0.0,0.228600,0.125604,0.079285,0.0,0.125604,0.270501,0.873820,0.270501
D5,0.0,0.0,0.284713,0.616455,0.207913,0.0,0.000000,0.284713,0.481917,0.427070
D6,0.0,0.0,0.198515,0.306210,0.124197,0.0,0.306210,0.076796,0.863622,0.076796
D7,0.0,0.0,0.280060,0.404253,0.144066,0.0,0.334352,0.093353,0.733792,0.280060
D8,0.0,0.0,0.430027,0.000000,0.209353,0.0,0.663318,0.406975,0.000000,0.406975
D9,0.0,0.0,0.128068,0.000000,0.118012,0.0,0.197545,0.000000,0.946281,0.187616


In [9]:
# Query as IDF-weighted binary vector, then L2-normalized (same style as many IR setups).
query_tfidf = query_vector * idf
query_norm = query_tfidf / norm(query_tfidf)

print("L2-normalized query vector (weights on query terms only):")
print(query_norm)

Normalized Query Vector:
t0    0.000000
t1    0.000000
t2    0.000000
t3    0.839096
t4    0.000000
t5    0.000000
t6    0.000000
t7    0.543983
t8    0.000000
t9    0.000000
dtype: float64


In [10]:
# Cosine similarity: for unit vectors, dot product equals cosine of the angle between them.
cosine_scores = tfidf_norm @ query_norm

results = pd.DataFrame({"Cosine_score": cosine_scores}, index=docs)
results = results.sort_values(by="Cosine_score", ascending=False)

print("Option 3: ranking by cosine similarity (highest first)")
results

Document Ranking (Cosine Similarity):


,Score
D18,0.800902
D5,0.672144
D2,0.657518
D3,0.584218
D10,0.567251
D17,0.528455
D19,0.494377
D11,0.424993
D1,0.415389
D7,0.389990


In [11]:
# Wide table for Option 3: same row order as the ranking above.
option3_output = tfidf_norm_df.copy()
option3_output.insert(0, "Cosine_score", cosine_scores)
option3_output = option3_output.reindex(results.index)

print("Option 3: cosine score + normalized TF-IDF per term (rows sorted by rank)")
option3_output

Final Output Table:


,Score,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9
D0,0.368609,0.0,0.0,0.268734,0.359473,0.087811,0.0,0.398048,0.123123,0.738262,0.268734
D1,0.415389,0.0,0.0,0.241440,0.320787,0.151868,0.0,0.254218,0.268791,0.817343,0.103983
D2,0.657518,0.0,0.0,0.139020,0.497910,0.000000,0.0,0.554315,0.440682,0.000000,0.480929
D3,0.584218,0.0,0.0,0.189101,0.516651,0.116168,0.0,0.291688,0.277028,0.640159,0.334943
D4,0.252542,0.0,0.0,0.228600,0.125604,0.079285,0.0,0.125604,0.270501,0.873820,0.270501
D5,0.672144,0.0,0.0,0.284713,0.616455,0.207913,0.0,0.000000,0.284713,0.481917,0.427070
D6,0.298715,0.0,0.0,0.198515,0.306210,0.124197,0.0,0.306210,0.076796,0.863622,0.076796
D7,0.389990,0.0,0.0,0.280060,0.404253,0.144066,0.0,0.334352,0.093353,0.733792,0.280060
D8,0.221388,0.0,0.0,0.430027,0.000000,0.209353,0.0,0.663318,0.406975,0.000000,0.406975
D9,0.000000,0.0,0.0,0.128068,0.000000,0.118012,0.0,0.197545,0.000000,0.946281,0.187616


### Explanation (from the ground up)

#### What problem are we solving?

Imagine a small **search engine**: you have many **documents** (texts represented as word counts) and a **query** (a few words). You want a **number per document** that says *how relevant* that document is to the query, then you **sort** documents from most to least relevant.

We never look at word order here; only **how many times each vocabulary term appears** in each document. That simplification is called a **bag of words**.

#### The input matrix

Rows = documents (`D0` … `D19`). Columns = terms (`t0` … `t9`). Each cell = **raw count** (term frequency in that document). Some columns are forced to be non-zero in every row so those words behave like *stopwords*: they appear everywhere and should not separate documents well later.

#### Turning counts into “TF” (term frequency)

Raw counts can dominate: a word that appears 100 times might not be “100× more important” than 10 times. A common fix is **log smoothing**: use $\log(1 + \text{count})$ via `np.log1p`. So repetition still matters, but with **diminishing returns**.

#### IDF (inverse document frequency)

**Document frequency** of term $t$: how many documents contain $t$ at least once. Terms that appear in **almost every** document are **not discriminating** (“the”, very common topics). Terms that appear in **few** documents are more **specific**.

We use $\text{idf}(t)=\log\frac{N}{n_t}$ where $N$ is the number of documents and $n_t$ is how many documents have $t$. If $t$ is in every document, $n_t=N$ and $\text{idf}(t)=0$: that term gets **no weight** for ranking (as in columns `t0`, `t1`, `t5` in our data).

#### TF-IDF

For each document $d$ and term $t$:

$$
\text{tfidf}(d,t) = \text{tf}(d,t)\times \text{idf}(t)
$$

So a term must both **show up in the document** (TF) and be **somewhat rare in the collection** (IDF) to get a large value. That is the main **Option 1** signal.

#### Option 1 — Query score (dot product)

The query lists a few terms. We encode it as a **binary vector**: 1 on those columns, 0 elsewhere. The **relevance score** of document $d$ is:

$$
\sum_{t} \text{tfidf}(d,t)\times q(t)
$$

which is exactly the **dot product** `tfidf @ query_vector`. It **adds** the TF-IDF weights of the query terms present in that document. Longer / heavier documents can score higher because we did **not** normalize yet.

The **Option 1 output table** uses this score in the first column and **raw TF-IDF** in the term columns (same order as the input).

#### Option 3 — Why normalize? Cosine similarity

If we only used the dot product, a very long document might get a larger score just because it has more words. **L2 normalization** divides each document vector by its Euclidean length so it becomes a **unit vector** (direction only, not magnitude). The **query** is built as IDF weights on the query terms and normalized the same way.

The **dot product of two unit vectors** equals the **cosine** of the angle between them (between $-1$ and $1$, here usually $\ge 0$). High cosine ⇒ the document’s term *profile* points in a similar direction to the query’s *profile*. That is **cosine similarity**.

We **sort** documents by cosine from **high to low** for **Option 3**. The wide table repeats those scores and shows the **normalized** TF-IDF in each term column, with rows in **rank order**.

#### Summary

| Step | Meaning |
|------|--------|
| Counts | What the data literally stores |
| Log TF | Don’t let huge counts explode |
| IDF | Down-weight terms that appear everywhere |
| TF-IDF | “Important in this doc” × “useful for distinguishing docs” |
| Option 1 dot product | Simple relevance = sum of query-term TF-IDFs |
| L2 norm + cosine | Fairer comparison of “shape”; rank by cosine |